In [15]:
import os
from torchvision import datasets,transforms
from torch.utils.data import DataLoader
from torch.utils.data import random_split
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import time
import torchvision.models as models
from matplotlib import pyplot as plt
import optuna

In [5]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [7]:
image_transform=transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2,contrast=0.2),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225]), 
])
dataset=datasets.ImageFolder(root="dataset/",transform=image_transform)
num_classes=len(dataset.classes)
num_classes
train_size=int(len(dataset)*0.75) 
val_size=len(dataset)-train_size


train_dataset,val_dataset=random_split(dataset,[train_size,val_size])

train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
val_loader=DataLoader(val_dataset,batch_size=32,shuffle=False)

In [10]:
class carclassifierresnet(nn.Module):
    def __init__(self, num_classes,dropout_rate=0.5):
        super().__init__()
        self.model = models.resnet50(weights="DEFAULT")
        # Freeze all layers
        for param in self.model.parameters():
            param.requires_grad = False
        # Unfreeze layer4
        for param in self.model.layer4.parameters():
            param.requires_grad = True
        # Get input features of final layer
        in_features = self.model.fc.in_features
        # Replace fully connected layer
        self.model.fc = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.model(x)

In [23]:
def objective(trial):
    lr=trial.suggest_float("lr",1e-5,1e-2,log=True)
    dropout_rate=trial.suggest_float("dropout_rate",0.2,0.7)
    model=carclassifierresnet(num_classes=num_classes,dropout_rate=dropout_rate).to(device)
    criterion=nn.CrossEntropyLoss()
    optimizer=optim.Adam(filter(lambda p : p.requires_grad,model.parameters()),lr=lr)
    epochs=3
    start=time.time()
    for epoch in range(epochs):
        model.train()
        running_loss=0.0
        for batch_num,(images,labels) in enumerate(train_loader):
            images,labels=images.to(device),labels.to(device)
            optimizer.zero_grad()
            outputs=model(images)
            loss=criterion(outputs,labels)
            loss.backward()
            optimizer.step()
            running_loss+=loss.item()*images.size(0) # total loss of an entire batch
        epoch_loss=running_loss/len(train_loader.dataset)
        

        model.eval()
        correct=0
        total=0
        all_labels=[]
        all_predicted=[]
        with torch.no_grad():
            for images,labels in val_loader:
                images,labels=images.to(device),labels.to(device)
                outputs=model(images)
                _,predicted=torch.max(outputs.data,1)
                total+=labels.size(0)
                correct+=(predicted==labels).sum().item()
                
        accuracy=(correct/total)*100
        trial.report(accuracy,epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
    end=time.time()
    print(f"Execution time :{end-start} seconds")
    return accuracy

In [24]:
study=optuna.create_study(direction="maximize")
study.optimize(objective,n_trials=20)

[I 2026-04-20 07:19:19,302] A new study created in memory with name: no-name-22f451ee-d48d-425c-b414-676b0d27dd22
[I 2026-04-20 07:24:19,770] Trial 0 finished with value: 52.77777777777778 and parameters: {'lr': 1.1247228066688242e-05, 'dropout_rate': 0.4640669941266525}. Best is trial 0 with value: 52.77777777777778.


Execution time :299.6313006877899 seconds


[I 2026-04-20 07:30:07,195] Trial 1 finished with value: 77.60416666666666 and parameters: {'lr': 0.001158392135810655, 'dropout_rate': 0.2771239464097069}. Best is trial 1 with value: 77.60416666666666.


Execution time :346.8960406780243 seconds


[I 2026-04-20 07:35:33,681] Trial 2 finished with value: 65.27777777777779 and parameters: {'lr': 2.4433541865319834e-05, 'dropout_rate': 0.3312021113351028}. Best is trial 1 with value: 77.60416666666666.


...

Execution time :322.7643301486969 seconds


[I 2026-04-20 07:57:03,021] Trial 7 pruned. 
[I 2026-04-20 07:59:27,032] Trial 8 pruned. 
[I 2026-04-20 08:02:38,717] Trial 9 pruned. 
[I 2026-04-20 08:05:48,307] Trial 10 pruned. 
[I 2026-04-20 08:12:10,005] Trial 11 pruned. 
[I 2026-04-20 08:14:50,101] Trial 12 pruned. 
[I 2026-04-20 08:16:49,065] Trial 13 pruned. 
[I 2026-04-20 08:18:45,616] Trial 14 pruned. 
[I 2026-04-20 08:20:40,865] Trial 15 pruned. 
[I 2026-04-20 08:26:30,808] Trial 16 finished with value: 75.86805555555556 and parameters: {'lr': 0.00039350661627966327, 'dropout_rate': 0.5080016193597573}. Best is trial 4 with value: 79.86111111111111.


Execution time :349.432963848114 seconds


[I 2026-04-20 08:28:28,583] Trial 17 pruned. 
[I 2026-04-20 08:30:33,371] Trial 18 pruned. 
[I 2026-04-20 08:36:22,088] Trial 19 finished with value: 77.08333333333334 and parameters: {'lr': 0.0024638840322144372, 'dropout_rate': 0.5256552729481451}. Best is trial 4 with value: 79.86111111111111.


Execution time :348.24072098731995 seconds


In [25]:
study.best_params

{'lr': 0.0013306375383660026, 'dropout_rate': 0.6729941640575401}